# D5 – Đánh giá trên tập test

Đầu vào: `models/` (TF-IDF, 3 mô hình, `calibrators.pkl`, `threshold.json`), `data/splits/val_b.csv`, `data/splits/test.csv`.
Đầu ra: `results/D5_metrics.csv`, `D5_paired.csv`, `D5_drift.csv`, `D5_prior_shift.csv`, `D5_top_features.csv`,
`D5_errors.csv`, `D5_confusion_matrix.png`, `D5_reliability.png`, `D5_tom_tat.txt`.

Quy tắc (README mục 7 – D5, mục 9; RULES.md; PROGRESS.md):
- `test.csv` chỉ được đọc trong notebook này và chạy **một lần**.
- **Không** huấn luyện lại, **không** fit lại TF-IDF hay bộ hiệu chuẩn, **không** chọn lại ngưỡng hay đổi ma trận chi phí.
- Bootstrap 1.000 lần, seed 42, chỉ lấy mẫu lại dự đoán đã có.
- Không in nội dung tin ra output; chỉ `results/D5_errors.csv` chứa tin ĐÃ CHE.

In [1]:
import json, os, pickle, platform, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from core import LABELS
from preprocess import transform_tfidf
from softmax import SoftmaxRegression
from calibration import get_calibrator
from calibration_metrics import plot_reliability
from evaluate import (METRICS, adjust_prior, bootstrap_indices, bootstrap_metrics, evaluate_probs,
                      nb_log_ratio, paired_difference, resample_by_prior, scenario_prior, summarize,
                      threshold_at_fpr, top_features)

SEED, N_BOOT, N_BINS = 42, 1000, 15
K = len(LABELS)
T_START = time.time()
SPLITS, MODELS, RESULTS = ROOT / "data" / "splits", ROOT / "models", ROOT / "results"
pd.set_option("display.width", 200)
MODEL_NAMES = ["softmax", "naive_bayes", "svm"]

def load_pkl(name):
    with open(MODELS / f"{name}.pkl", "rb") as f:
        return pickle.load(f)

vec_word, vec_char = load_pkl("tfidf_word"), load_pkl("tfidf_char")
nb, svm, calibrators = load_pkl("naive_bayes"), load_pkl("svm"), load_pkl("calibrators")
sm = SoftmaxRegression.load(MODELS / "softmax.npz")
with open(MODELS / "threshold.json", encoding="utf-8") as f:
    THR = json.load(f)
C = np.array(THR["cost_matrix"], dtype=float)
PI_TRAIN = np.array(THR["pi_train"])
assert THR["labels"] == LABELS
print("C =\n", C, "\nπ_train =", PI_TRAIN.round(4))
print({m: (THR["models"][m]["calibrator"], round(THR["models"][m]["threshold"], 4)) for m in MODEL_NAMES})

def scores(m, X):
    # Điểm (n, K) đưa vào bộ hiệu chuẩn (RULES.md).
    return {"softmax": sm.decision_function, "naive_bayes": nb.predict_log_proba,
            "svm": svm.decision_function}[m](X)

def load_split(name):
    d = pd.read_csv(SPLITS / f"{name}.csv", encoding="utf-8-sig")
    d["y"] = d["label"].map({l: i for i, l in enumerate(LABELS)}).astype(int)
    return d

C =
 [[ 0.  5. 10.]
 [ 1.  0.  1.]
 [ 5.  3.  0.]] 
π_train = [0.2866 0.3792 0.3343]
{'softmax': ('temperature', 0.7076), 'naive_bayes': ('none', 0.9038), 'svm': ('temperature', 0.6597)}


## 1. Kiểm tra trên val_b (trước khi đọc test)

Nạp lại mô hình phải cho đúng các số D4 đã ghi: log-loss/ECE trong `results/D4_hieu_chuan.csv` (dòng được chọn)
và ngưỡng trong `threshold.json` (tính lại bằng đúng quy tắc của D4).

"Trước hiệu chuẩn" = softmax của điểm (`NoCalibrator`): softmax và NB là xác suất gốc; SVM chỉ để tham khảo.
Ngưỡng FPR 1% của "trước hiệu chuẩn" **không có** trong `threshold.json`, nên được tính trên **val_b** bằng
đúng quy tắc D4 (chỉ để tham chiếu; ngưỡng chính thức là ngưỡng sau hiệu chuẩn trong `threshold.json`).

In [2]:
val_b = load_split("val_b")
X_vb, y_vb = transform_tfidf(vec_word, vec_char, val_b), val_b.y.to_numpy()
none_cal = get_calibrator("none", K)
P_vb = {}
for m in MODEL_NAMES:
    S = scores(m, X_vb)
    P_vb[(m, "truoc")] = none_cal.predict_proba(S)
    P_vb[(m, "sau")] = calibrators[m].predict_proba(S)

hc = pd.read_csv(RESULTS / "D4_hieu_chuan.csv", encoding="utf-8-sig").query("chon == 1").set_index("mo_hinh")
THRESH = {}
for m in MODEL_NAMES:
    info = THR["models"][m]
    assert calibrators[m].name == info["calibrator"] == hc.loc[m, "phuong_phap"]
    t, fpr, rec = threshold_at_fpr(P_vb[(m, "sau")][:, 0], y_vb == 0, THR["fpr_muc_tieu"])
    assert np.isclose(t, info["threshold"]) and np.isclose(rec, info["recall_lua_dao_val_b"])
    met = evaluate_probs(P_vb[(m, "sau")], y_vb, C, info["threshold"], N_BINS)
    assert np.isclose(met["log_loss"], hc.loc[m, "logloss_val_b"]) and np.isclose(met["ece_top_label"], hc.loc[m, "ece_top_val_b"])
    THRESH[(m, "sau")] = info["threshold"]
    t0, fpr0, rec0 = threshold_at_fpr(P_vb[(m, "truoc")][:, 0], y_vb == 0, THR["fpr_muc_tieu"])
    THRESH[(m, "truoc")] = t0
    print(f"{m:12s} sau: t = {info['threshold']:.4f} (khớp D4) | trước (tham chiếu, val_b): t = {t0:.4f}, "
          f"FPR = {fpr0:.4f}, recall = {rec0:.4f}")
print("Nạp lại mô hình khớp D4 trên val_b.")

softmax      sau: t = 0.7076 (khớp D4) | trước (tham chiếu, val_b): t = 0.6244, FPR = 0.0078, recall = 0.7907
naive_bayes  sau: t = 0.9038 (khớp D4) | trước (tham chiếu, val_b): t = 0.9038, FPR = 0.0078, recall = 0.1860
svm          sau: t = 0.6597 (khớp D4) | trước (tham chiếu, val_b): t = 0.4302, FPR = 0.0078, recall = 0.8140
Nạp lại mô hình khớp D4 trên val_b.


## 2. Dự đoán trên test (đọc test.csv MỘT lần)

In [3]:
test = load_split("test")
X_te, y_te = transform_tfidf(vec_word, vec_char, test), test.y.to_numpy()
assert not set(test.id) & set(val_b.id), "test và val_b chồng lấn"
print("test:", X_te.shape, "| số tin theo lớp:", dict(zip(LABELS, np.bincount(y_te, minlength=K))))
print("nguồn:", test.nguon.value_counts().to_dict())
def drange(d):
    t = pd.to_datetime(d.date, format="%m/%d/%Y")
    return f"{t.min().date()} → {t.max().date()}"
print("date test:", drange(test), "| date val_b:", drange(val_b))
P_te = {}
for m in MODEL_NAMES:
    S = scores(m, X_te)
    P_te[(m, "truoc")] = none_cal.predict_proba(S)
    P_te[(m, "sau")] = calibrators[m].predict_proba(S)
CONFIGS = [(m, ph) for m in MODEL_NAMES for ph in ("truoc", "sau")]

test: (355, 20586) | số tin theo lớp: {'lua_dao': np.int64(97), 'rac': np.int64(139), 'hop_le': np.int64(119)}
nguồn: {'tran2026': 355}
date test: 2026-07-24 → 2026-08-03 | date val_b: 2026-07-24 → 2026-08-03


## 3. Chỉ số + bootstrap 1.000 lần → D5_metrics.csv

Mỗi chỉ số: giá trị gốc, mean ± std, KTC 95% (phân vị 2,5–97,5) qua 1.000 mẫu bootstrap theo tin nhắn
(seed 42, cùng một bộ chỉ số mẫu cho mọi mô hình → dùng lại cho bootstrap ghép cặp).
F1 macro, chi phí và recall theo lớp tính theo quy tắc argmin_j Σ_i C[i,j]·p(i|x);
recall lừa đảo @FPR dùng ngưỡng cố định (lừa đảo so với phần còn lại).

In [4]:
t0 = time.time()
IDX = bootstrap_indices(len(y_te), N_BOOT, SEED)
POINT, BOOT, rows = {}, {}, []
for m, ph in CONFIGS:
    POINT[(m, ph)] = evaluate_probs(P_te[(m, ph)], y_te, C, THRESH[(m, ph)], N_BINS)
    BOOT[(m, ph)] = bootstrap_metrics(P_te[(m, ph)], y_te, C, THRESH[(m, ph)], IDX, N_BINS)
    s = summarize(POINT[(m, ph)], BOOT[(m, ph)])
    s.insert(0, "phuong_phap", "none" if ph == "truoc" else calibrators[m].name)
    s.insert(0, "hieu_chuan", ph)
    s.insert(0, "mo_hinh", m)
    s["nguong"] = THRESH[(m, ph)]
    rows.append(s)
metrics_df = pd.concat(rows, ignore_index=True)
metrics_df["n_test"], metrics_df["n_boot"], metrics_df["seed"] = len(y_te), N_BOOT, SEED
metrics_df.to_csv(RESULTS / "D5_metrics.csv", index=False, encoding="utf-8-sig")
print(f"bootstrap: {time.time() - t0:.1f} s")

def fmt_ci(r):
    return f"{r.goc:.3f} | {r['mean']:.3f} ± {r['std']:.3f} | [{r.ci_low:.3f}; {r.ci_high:.3f}]"
view = metrics_df[metrics_df.chi_so.isin(METRICS + ["bao_nham_lua_dao"])].copy()
view["gia_tri"] = view.apply(fmt_ci, axis=1)
print(view.pivot_table(index=["mo_hinh", "hieu_chuan"], columns="chi_so", values="gia_tri", aggfunc="first")
      .reindex(columns=METRICS + ["bao_nham_lua_dao"]).T.to_string())

bootstrap: 8.2 s
mo_hinh                                           naive_bayes                                                                         softmax                                                                             svm                                        
hieu_chuan                                                sau                                   truoc                                     sau                                   truoc                                     sau                                   truoc
chi_so                                                                                                                                                                                                                                                               
f1_macro               0.843 | 0.844 ± 0.021 | [0.802; 0.883]  0.843 | 0.844 ± 0.021 | [0.802; 0.883]  0.897 | 0.898 ± 0.017 | [0.863; 0.928]  0.885 | 0.886 ± 0.017 | [0.852; 0.918]  0.849 | 0.849 

## 4. Bootstrap ghép cặp (sau hiệu chuẩn) → D5_paired.csv

Hiệu = softmax − mô hình kia, trên cùng các mẫu bootstrap. Chỉ kết luận khác biệt khi KTC 95% của hiệu không chứa 0.

In [5]:
pair_rows = []
for other in ["naive_bayes", "svm"]:
    a, b = ("softmax", "sau"), (other, "sau")
    d = paired_difference(POINT[a], POINT[b], BOOT[a], BOOT[b])
    d.insert(0, "so_sanh", f"softmax - {other}")
    pair_rows.append(d)
paired_df = pd.concat(pair_rows, ignore_index=True)
paired_df.to_csv(RESULTS / "D5_paired.csv", index=False, encoding="utf-8-sig")
print(paired_df[paired_df.chi_so.isin(METRICS)].to_string(index=False, float_format=lambda v: f"{v:.4f}"))

              so_sanh                chi_so  hieu_goc    mean    std  ci_low  ci_high  n_hop_le  ci_chua_0
softmax - naive_bayes              f1_macro    0.0537  0.0539 0.0185  0.0173   0.0900      1000      False
softmax - naive_bayes            chi_phi_tb   -0.4028 -0.4043 0.1110 -0.6310  -0.2000      1000      False
softmax - naive_bayes recall_lua_dao_at_fpr    0.5464  0.5487 0.0511  0.4476   0.6471      1000      False
softmax - naive_bayes      fpr_at_threshold    0.0116  0.0119 0.0066  0.0000   0.0268      1000       True
softmax - naive_bayes         ece_top_label   -0.0575 -0.0534 0.0153 -0.0830  -0.0229      1000      False
softmax - naive_bayes           ece_lua_dao   -0.0561 -0.0529 0.0129 -0.0781  -0.0278      1000      False
softmax - naive_bayes              log_loss   -0.2110 -0.2108 0.0399 -0.2875  -0.1369      1000      False
        softmax - svm              f1_macro    0.0486  0.0485 0.0126  0.0245   0.0742      1000      False
        softmax - svm            chi_

## 5. Trôi khái niệm: val_b so với test → D5_drift.csv

Cùng mô hình, cùng bộ hiệu chuẩn, cùng ngưỡng. val_b và test là hai mẫu độc lập nên KTC của hiệu (test − val_b)
lấy từ hai bootstrap độc lập (val_b dùng seed 43 để không trùng chỉ số với test).
Lưu ý: split phân tầng theo lớp nên tỉ lệ lớp của val_b và test gần nhau; ở đây chỉ đo được trôi **trong từng lớp**,
không đo được thay đổi tỉ lệ lớp theo thời gian.

In [6]:
IDX_VB = bootstrap_indices(len(y_vb), N_BOOT, SEED + 1)
drift_rows = []
for m in MODEL_NAMES:
    key = (m, "sau")
    pv = evaluate_probs(P_vb[key], y_vb, C, THRESH[key], N_BINS)
    bv = bootstrap_metrics(P_vb[key], y_vb, C, THRESH[key], IDX_VB, N_BINS)
    diff = summarize({k: POINT[key][k] - pv[k] for k in pv}, BOOT[key] - bv)
    for r in diff.itertuples():
        drift_rows.append(dict(mo_hinh=m, hieu_chuan=calibrators[m].name, chi_so=r.chi_so,
                               val_b=pv[r.chi_so], test=POINT[key][r.chi_so], chenh_test_tru_val_b=r.goc,
                               chenh_ci_low=r.ci_low, chenh_ci_high=r.ci_high,
                               ci_chua_0=bool(r.ci_low <= 0 <= r.ci_high)))
drift_df = pd.DataFrame(drift_rows)
drift_df.to_csv(RESULTS / "D5_drift.csv", index=False, encoding="utf-8-sig")
print(drift_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

    mo_hinh  hieu_chuan                chi_so  val_b   test  chenh_test_tru_val_b  chenh_ci_low  chenh_ci_high  ci_chua_0
    softmax temperature              f1_macro 0.8942 0.8971                0.0029       -0.0491         0.0624       True
    softmax temperature            chi_phi_tb 0.3605 0.5211                0.1607       -0.1594         0.4294       True
    softmax temperature recall_lua_dao_at_fpr 0.7907 0.7835               -0.0072       -0.1429         0.1495       True
    softmax temperature      fpr_at_threshold 0.0078 0.0155                0.0078       -0.0125         0.0277       True
    softmax temperature         ece_top_label 0.0445 0.0400               -0.0045       -0.0537         0.0261       True
    softmax temperature           ece_lua_dao 0.0353 0.0379                0.0027       -0.0308         0.0273       True
    softmax temperature              log_loss 0.2387 0.2773                0.0386       -0.0868         0.1481       True
    softmax temperature 

## 6. Ma trận nhầm lẫn và biểu đồ độ tin cậy

In [7]:
CM = {}
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, m in zip(axes, MODEL_NAMES):
    pred = np.argmin(P_te[(m, "sau")] @ C, axis=1)
    cm = confusion_matrix(y_te, pred, labels=list(range(K)))
    CM[m] = cm
    ax.imshow(cm, cmap="Blues")
    for i in range(K):
        for j in range(K):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks(range(K)); ax.set_xticklabels(LABELS); ax.set_yticks(range(K)); ax.set_yticklabels(LABELS)
    ax.set_xlabel("Lớp đoán (quy tắc chi phí)"); ax.set_ylabel("Lớp thật")
    ax.set_title(f"{m} ({calibrators[m].name})")
fig.tight_layout()
fig.savefig(RESULTS / "D5_confusion_matrix.png", dpi=150)
plt.close(fig)
for m in MODEL_NAMES:
    print(m, "\n", pd.DataFrame(CM[m], index=LABELS, columns=LABELS))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for j, m in enumerate(MODEL_NAMES):
    for i, (kind, title) in enumerate([("top_label", "top-label"), ("binary", "lớp lừa đảo")]):
        ax = axes[i, j]
        for ph, lbl in [("truoc", "trước (none)"), ("sau", f"sau ({calibrators[m].name})")]:
            if ph == "sau" and calibrators[m].name == "none":
                continue
            P = P_te[(m, ph)]
            if kind == "top_label":
                plot_reliability(P, y_te, N_BINS, "top_label", ax=ax, label=lbl, show_hist=(ph == "truoc"))
            else:
                plot_reliability(P[:, 0], (y_te == 0).astype(int), N_BINS, "binary", ax=ax, label=lbl,
                                 show_hist=(ph == "truoc"))
        ax.set_title(f"{m}: {title}")
        if kind == "binary":
            ax.legend(loc="lower right", fontsize="small")
fig.tight_layout()
fig.savefig(RESULTS / "D5_reliability.png", dpi=150)
plt.close(fig)

softmax 
          lua_dao  rac  hop_le
lua_dao       87    0      10
rac            2  131       6
hop_le        13    4     102
naive_bayes 
          lua_dao  rac  hop_le
lua_dao       72    3      22
rac            2  132       5
hop_le        13    7      99
svm 
          lua_dao  rac  hop_le
lua_dao       83    1      13
rac            3  130       6
hop_le        24    4      91


## 7. Điều chỉnh tỷ lệ lớp (B6) → D5_prior_shift.csv

p′(y|x) ∝ p(y|x)·π′_y/π_y, π = tỉ lệ lớp của train (`threshold.json`).
Kịch bản: lừa đảo chiếm 5% và 1%; rác và hợp lệ giữ tỉ lệ với nhau như train.
Mô phỏng: 1.000 lần (seed 42) lấy mẫu lại test theo π′ (trong từng lớp, có hoàn lại), cỡ mẫu = số tin test;
trên cùng một mẫu, so xác suất đã hiệu chuẩn chưa điều chỉnh và đã điều chỉnh (ghép cặp).
Quyết định theo quy tắc chi phí với cùng ma trận C; ngưỡng FPR không dùng ở đây.

In [8]:
PS_METRICS = ["log_loss", "ece_top_label", "ece_lua_dao", "chi_phi_tb", "f1_macro",
              "recall_lua_dao", "bao_nham_lua_dao"]
t0 = time.time()
ps_rows = []
for p_fraud in [0.05, 0.01]:
    pi_new = scenario_prior(PI_TRAIN, p_fraud)
    rng = np.random.default_rng(SEED)
    idx_list = [resample_by_prior(y_te, pi_new, len(y_te), rng) for _ in range(N_BOOT)]
    for m in MODEL_NAMES:
        P0 = P_te[(m, "sau")]
        P1 = adjust_prior(P0, PI_TRAIN, pi_new)
        b0 = pd.DataFrame([evaluate_probs(P0[i], y_te[i], C, None, N_BINS) for i in idx_list])
        b1 = pd.DataFrame([evaluate_probs(P1[i], y_te[i], C, None, N_BINS) for i in idx_list])
        for k in PS_METRICS:
            v0, v1 = b0[k].to_numpy(float), b1[k].to_numpy(float)
            ok = ~np.isnan(v0) & ~np.isnan(v1)
            d = v1[ok] - v0[ok]
            ps_rows.append(dict(kich_ban=f"lua_dao {p_fraud:.0%}", pi_moi=np.round(pi_new, 4).tolist(), mo_hinh=m,
                                chi_so=k, truoc_mean=v0[ok].mean(), truoc_std=v0[ok].std(ddof=1),
                                sau_mean=v1[ok].mean(), sau_std=v1[ok].std(ddof=1),
                                hieu_mean=d.mean(), hieu_ci_low=np.percentile(d, 2.5),
                                hieu_ci_high=np.percentile(d, 97.5), n_hop_le=int(ok.sum())))
ps_df = pd.DataFrame(ps_rows)
ps_df["n_mau"], ps_df["n_lap"], ps_df["seed"] = len(y_te), N_BOOT, SEED
ps_df.to_csv(RESULTS / "D5_prior_shift.csv", index=False, encoding="utf-8-sig")
print(f"prior shift: {time.time() - t0:.1f} s")
print(ps_df.drop(columns=["pi_moi", "n_mau", "n_lap", "seed"]).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

prior shift: 18.1 s
  kich_ban     mo_hinh           chi_so  truoc_mean  truoc_std  sau_mean  sau_std  hieu_mean  hieu_ci_low  hieu_ci_high  n_hop_le
lua_dao 5%     softmax         log_loss      0.2383     0.0383    0.2060   0.0342    -0.0323      -0.0590       -0.0082      1000
lua_dao 5%     softmax    ece_top_label      0.0424     0.0101    0.0386   0.0097    -0.0038      -0.0232        0.0162      1000
lua_dao 5%     softmax      ece_lua_dao      0.0466     0.0075    0.0222   0.0052    -0.0244      -0.0382       -0.0103      1000
lua_dao 5%     softmax       chi_phi_tb      0.3716     0.0687    0.2407   0.0565    -0.1309      -0.2563       -0.0113      1000
lua_dao 5%     softmax         f1_macro      0.8162     0.0249    0.8873   0.0285     0.0711       0.0161        0.1207      1000
lua_dao 5%     softmax   recall_lua_dao      0.8979     0.0685    0.7332   0.0992    -0.1647      -0.3333        0.0000      1000
lua_dao 5%     softmax bao_nham_lua_dao      0.0593     0.0122    0.00

## 8. Đặc trưng có trọng số lớn → D5_top_features.csv

Softmax: cột W[:, k]; SVM: coef_[k] (one-vs-rest); NB: log p(w|k) − log(trung bình p(w|j), j ≠ k).
Mỗi mô hình, mỗi lớp: 10 n-gram từ + 10 n-gram ký tự có trọng số lớn nhất (không in ra output).

In [9]:
names_w = np.array(["tu: " + s for s in vec_word.get_feature_names_out()])
names_c = np.array(["ky_tu: " + s for s in vec_char.get_feature_names_out()])
d_w = len(names_w)
assert d_w + len(names_c) == sm.W_.shape[0] == svm.coef_.shape[1]
WEIGHTS = {"softmax": sm.W_, "svm": svm.coef_.T, "naive_bayes": nb_log_ratio(nb.feature_log_prob_)}
tf_rows = []
for m, Wm in WEIGHTS.items():
    for loai, sl, nm in [("tu", slice(0, d_w), names_w), ("ky_tu", slice(d_w, None), names_c)]:
        t = top_features(Wm[sl], nm, top=10)
        t.insert(0, "loai_ngram", loai)
        t.insert(0, "mo_hinh", m)
        tf_rows.append(t)
top_df = pd.concat(tf_rows, ignore_index=True)
top_df.to_csv(RESULTS / "D5_top_features.csv", index=False, encoding="utf-8-sig")
print("D5_top_features.csv:", len(top_df), "dòng; d_tu =", d_w, ", d_ky_tu =", len(names_c))

D5_top_features.csv: 180 dòng; d_tu = 10819 , d_ky_tu = 9767


## 9. Phân tích lỗi → D5_errors.csv (tin ĐÃ CHE)

Mỗi dòng là một tin bị đoán sai (sau hiệu chuẩn, quy tắc chi phí) bởi một mô hình. Output notebook chỉ in số đếm.

In [10]:
err_rows = []
for m in MODEL_NAMES:
    P = P_te[(m, "sau")]
    pred = np.argmin(P @ C, axis=1)
    wrong = pred != y_te
    e = test.loc[wrong, ["id", "date", "nguon", "nhom_gan_trung", "label"]].copy()
    e.insert(0, "mo_hinh", m)
    e["nhan_doan"] = [LABELS[k] for k in pred[wrong]]
    for k, l in enumerate(LABELS):
        e[f"p_{l}"] = P[wrong, k]
    e["chi_phi"] = C[y_te[wrong], pred[wrong]]
    e["text_clean"] = test.loc[wrong, "text_clean"].to_numpy()
    err_rows.append(e.rename(columns={"label": "nhan_that"}))
err_df = pd.concat(err_rows, ignore_index=True)
err_df.to_csv(RESULTS / "D5_errors.csv", index=False, encoding="utf-8-sig")
print("Số lỗi theo mô hình × (thật → đoán):")
print(err_df.groupby(["mo_hinh", "nhan_that", "nhan_doan"]).size().unstack("mo_hinh", fill_value=0))
print("\nSố lỗi theo mô hình × lớp thật × nguồn:")
print(err_df.groupby(["mo_hinh", "nhan_that", "nguon"]).size().unstack("mo_hinh", fill_value=0))
ids = {m: set(err_df.loc[err_df.mo_hinh == m, "id"]) for m in MODEL_NAMES}
print("\nTin sai ở cả 3 mô hình:", len(ids["softmax"] & ids["naive_bayes"] & ids["svm"]))

Số lỗi theo mô hình × (thật → đoán):
mo_hinh              naive_bayes  softmax  svm
nhan_that nhan_doan                           
hop_le    lua_dao             13       13   24
          rac                  7        4    4
lua_dao   hop_le              22       10   13
          rac                  3        0    1
rac       hop_le               5        6    6
          lua_dao              2        2    3

Số lỗi theo mô hình × lớp thật × nguồn:
mo_hinh             naive_bayes  softmax  svm
nhan_that nguon                              
hop_le    tran2026           20       17   28
lua_dao   tran2026           25       10   14
rac       tran2026            7        8    9

Tin sai ở cả 3 mô hình: 25


## 10. Tóm tắt (results/D5_tom_tat.txt)

In [11]:
def machine_info():
    ram = None
    if os.name == "nt":
        import ctypes
        class MS(ctypes.Structure):
            _fields_ = [("dwLength", ctypes.c_ulong), ("dwMemoryLoad", ctypes.c_ulong),
                        ("ullTotalPhys", ctypes.c_ulonglong), ("ullAvailPhys", ctypes.c_ulonglong),
                        ("ullTotalPageFile", ctypes.c_ulonglong), ("ullAvailPageFile", ctypes.c_ulonglong),
                        ("ullTotalVirtual", ctypes.c_ulonglong), ("ullAvailVirtual", ctypes.c_ulonglong),
                        ("sullAvailExtendedVirtual", ctypes.c_ulonglong)]
        ms = MS(); ms.dwLength = ctypes.sizeof(MS)
        ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(ms))
        ram = ms.ullTotalPhys
    ram_s = f"{ram / 2**30:.1f} GB" if ram else "không xác định"
    return (f"CPU: {platform.processor() or platform.machine()} ({os.cpu_count()} luồng logic)\nRAM: {ram_s}\n"
            f"OS: {platform.platform()}\nPython {platform.python_version()} | numpy {np.__version__} | "
            f"scikit-learn {sklearn.__version__} | pandas {pd.__version__}")

out = []
p = out.append
p(f"test: {len(y_te)} tin, theo lớp {dict(zip(LABELS, np.bincount(y_te, minlength=K).tolist()))}; "
  f"date {drange(test)} (val_b {drange(val_b)}); nguồn {test.nguon.value_counts().to_dict()}")
p(f"Ngưỡng: {json.dumps({f'{m}|{ph}': THRESH[(m, ph)] for m, ph in CONFIGS})}")
p("\n=== D5_metrics (giá trị gốc | mean ± std | KTC 95%) ===")
p(view.pivot_table(index=["mo_hinh", "hieu_chuan"], columns="chi_so", values="gia_tri", aggfunc="first")
  .reindex(columns=METRICS + ["bao_nham_lua_dao"]).T.to_string())
p("\n=== recall theo lớp (quy tắc chi phí) ===")
p(metrics_df[metrics_df.chi_so.str.startswith("recall_") & (metrics_df.chi_so != "recall_lua_dao_at_fpr")]
  .assign(gia_tri=lambda d: d.apply(fmt_ci, axis=1))
  .pivot_table(index=["mo_hinh", "hieu_chuan"], columns="chi_so", values="gia_tri", aggfunc="first").T.to_string())
p("\n=== D5_paired ===")
p(paired_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
p("\n=== D5_drift ===")
p(drift_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
p("\n=== D5_prior_shift ===")
p(ps_df.drop(columns=["n_mau", "n_lap", "seed"]).to_string(index=False, float_format=lambda v: f"{v:.4f}"))
p("\n=== Ma trận nhầm lẫn test (hàng = thật, cột = đoán; " + ", ".join(LABELS) + ") ===")
for m in MODEL_NAMES:
    p(f"[{m}]\n" + pd.DataFrame(CM[m], index=LABELS, columns=LABELS).to_string())
p("\n=== Lỗi theo mô hình × (thật → đoán) ===")
p(err_df.groupby(["mo_hinh", "nhan_that", "nhan_doan"]).size().unstack("mo_hinh", fill_value=0).to_string())
p("\n=== Cấu hình máy ===")
p(machine_info())
p(f"Tổng thời gian chạy notebook: {time.time() - T_START:.1f} s")
summary = "\n".join(out)
(RESULTS / "D5_tom_tat.txt").write_text(summary, encoding="utf-8")
print(summary)

test: 355 tin, theo lớp {'lua_dao': 97, 'rac': 139, 'hop_le': 119}; date 2026-07-24 → 2026-08-03 (val_b 2026-07-24 → 2026-08-03); nguồn {'tran2026': 355}
Ngưỡng: {"softmax|truoc": 0.624403739382046, "softmax|sau": 0.7076425458887482, "naive_bayes|truoc": 0.9038263562879011, "naive_bayes|sau": 0.9038263562879011, "svm|truoc": 0.4301873436467626, "svm|sau": 0.6597216018333596}

=== D5_metrics (giá trị gốc | mean ± std | KTC 95%) ===
mo_hinh                                           naive_bayes                                                                         softmax                                                                             svm                                        
hieu_chuan                                                sau                                   truoc                                     sau                                   truoc                                     sau                                   truoc
chi_so                                   